In [ ]:
import json
from collections import Counter
from pathlib import Path
from typing import Any


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


chunks = load_jsonl(
    Path("data/processed/chunks/sec_10k_chunks.jsonl")
)
proxies = load_jsonl(
    Path(
        "data/processed/table_proxies/"
        "sec_10k_table_proxies.jsonl"
    )
)
records = load_jsonl(
    Path(
        "data/processed/embedding_records/"
        "sec_10k_embedding_records.jsonl"
    )
)

chunks_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in chunks
}
proxies_by_target = {
    proxy["target_chunk_id"]: proxy
    for proxy in proxies
}

type_counts = Counter(
    record["record_type"]
    for record in records
)

assert type_counts == {"text": 3890, "table": 3220}
assert len(records) == 7110
assert len({record["record_id"] for record in records}) == 7110

for record in records:
    assert record["record_type"] in {"text", "table"}
    assert record["embedding_text"].strip()
    assert record["document"].strip()
    assert record["target_chunk_id"] in chunks_by_id

    assert all(
        isinstance(value, (str, int, float, bool))
        for value in record["metadata"].values()
    )

    chunk = chunks_by_id[record["target_chunk_id"]]

    assert record["document"] == chunk["text"]
    assert "SEC item: None" not in record["embedding_text"]
    assert "Section: None" not in record["embedding_text"]

    if record["record_type"] == "text":
        assert record["record_id"] == f"text::{chunk['chunk_id']}"
        assert record["embedding_text"].endswith(chunk["text"])

    else:
        proxy = proxies_by_target[record["target_chunk_id"]]

        assert record["record_id"] == f"table::{proxy['proxy_id']}"
        assert record["embedding_text"] == proxy["proxy_text"]
        assert chunk["element_type"] == "table"

print("All embedding record checks passed!")
print(f"Record types: {dict(type_counts)}")
print(
    "Banks:",
    sorted({record["metadata"]["ticker"] for record in records}),
)

In [ ]:
sample_specs = [
    ("text", "JPM"),
    ("text", "BAC"),
    ("table", "GS"),
    ("table", "PNC"),
]

for record_type, ticker in sample_specs:
    record = next(
        record
        for record in records
        if record["record_type"] == record_type
        and record["metadata"]["ticker"] == ticker
    )

    print("\n" + "=" * 80)
    print(f"{record_type.upper()} | {ticker}")
    print(f"Record ID: {record['record_id']}")
    print(f"Target: {record['target_chunk_id']}")
    print("\nEMBEDDING TEXT:")
    print(record["embedding_text"][:600])
    print("\nORIGINAL DOCUMENT:")
    print(record["document"][:600])

In [ ]:
from collections import Counter


def document_word_count(record: dict) -> int:
    return len(record["document"].split())


thresholds = (5, 10, 25, 50)

for record_type in ("text", "table"):
    type_records = [
        record
        for record in records
        if record["record_type"] == record_type
    ]

    print(f"\n{record_type.upper()}: {len(type_records)}")

    for threshold in thresholds:
        count = sum(
            document_word_count(record) <= threshold
            for record in type_records
        )
        percentage = 100 * count / len(type_records)

        print(
            f"At most {threshold:>2} words: "
            f"{count} ({percentage:.1f}%)"
        )

    shortest_sections = Counter(
        str(record["metadata"].get("section_title", ""))
        for record in type_records
        if document_word_count(record) <= 10
    )

    print("Most common sections among records with at most 10 words:")

    for section, count in shortest_sections.most_common(10):
        print(f"  {count:>4} | {section or '<missing>'}")

In [ ]:
sample_specs = [
    ("text", "JPM", 80),
    ("text", "BAC", 80),
    ("table", "GS", 30),
    ("table", "PNC", 30),
]

for record_type, ticker, minimum_words in sample_specs:
    candidates = [
        record
        for record in records
        if record["record_type"] == record_type
        and record["metadata"]["ticker"] == ticker
        and document_word_count(record) >= minimum_words
    ]

    record = candidates[len(candidates) // 2]

    print("\n" + "=" * 80)
    print(f"{record_type.upper()} | {ticker}")
    print("Section:", record["metadata"].get("section_title"))
    print("Document words:", document_word_count(record))

    print("\nEMBEDDING TEXT:")
    print(record["embedding_text"][:800])

    print("\nORIGINAL DOCUMENT:")
    print(record["document"][:800])

In [ ]:
from collections import Counter
from time import perf_counter

import numpy as np
import torch
from sentence_transformers import SentenceTransformer


MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
BATCH_SIZE = 4


def embedding_word_count(record: dict) -> int:
    return len(record["embedding_text"].split())


# Select one representative record per bank and record type.
smoke_records_by_id = {}

tickers = sorted(
    {
        record["metadata"]["ticker"]
        for record in records
    }
)

for ticker in tickers:
    for record_type in ("text", "table"):
        candidates = sorted(
            (
                record
                for record in records
                if record["record_type"] == record_type
                and record["metadata"]["ticker"] == ticker
            ),
            key=embedding_word_count,
        )

        if candidates:
            representative = candidates[len(candidates) // 2]
            smoke_records_by_id[representative["record_id"]] = representative

# Add the longest records to test memory usage and long inputs.
for record in sorted(
    records,
    key=embedding_word_count,
    reverse=True,
)[:8]:
    smoke_records_by_id[record["record_id"]] = record

smoke_records = list(smoke_records_by_id.values())
embedding_texts = [
    record["embedding_text"]
    for record in smoke_records
]

print(f"Smoke records: {len(smoke_records)}")
print(
    "Record types:",
    dict(Counter(record["record_type"] for record in smoke_records)),
)
print(
    "Banks:",
    sorted(
        {
            record["metadata"]["ticker"]
            for record in smoke_records
        }
    ),
)
print(
    "Embedding words:",
    f"min={min(map(embedding_word_count, smoke_records))},",
    f"max={max(map(embedding_word_count, smoke_records))}",
)

model = SentenceTransformer(MODEL_NAME)

if model.device.type == "cuda":
    model.half()
    torch.cuda.reset_peak_memory_stats()

print(f"Device: {model.device}")
print(f"Maximum sequence length: {model.max_seq_length}")

started_at = perf_counter()

embeddings = model.encode_document(
    embedding_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

elapsed_seconds = perf_counter() - started_at
embeddings = np.asarray(embeddings, dtype=np.float32)
embedding_dimension = model.get_sentence_embedding_dimension()

assert embedding_dimension is not None
assert embeddings.shape == (
    len(smoke_records),
    embedding_dimension,
)
assert embedding_dimension == 1024
assert np.isfinite(embeddings).all()

norms = np.linalg.norm(embeddings, axis=1)

assert np.allclose(norms, 1.0, atol=1e-5)

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(
    "Norms:",
    f"min={norms.min():.6f},",
    f"mean={norms.mean():.6f},",
    f"max={norms.max():.6f}",
)
print(f"Time: {elapsed_seconds:.1f} s")

if model.device.type == "cuda":
    peak_memory_gb = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )
    print(f"Peak allocated GPU memory: {peak_memory_gb:.2f} GB")

print("Qwen document embedding smoke test passed!")

In [ ]:
norm_deviations = np.abs(norms - 1.0)
worst_indices = np.argsort(norm_deviations)[-5:][::-1]

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(
    "Norms:",
    f"min={norms.min():.8f},",
    f"mean={norms.mean():.8f},",
    f"max={norms.max():.8f}",
)
print(f"Maximum deviation: {norm_deviations.max():.8f}")

print("\nLargest deviations:")

for index in worst_indices:
    record = smoke_records[index]

    print(
        f"{index:>2} | "
        f"norm={norms[index]:.8f} | "
        f"type={record['record_type']} | "
        f"bank={record['metadata']['ticker']} | "
        f"words={embedding_word_count(record)}"
    )

assert embeddings.shape == (len(smoke_records), 1024)
assert embeddings.dtype == np.float32
assert np.isfinite(embeddings).all()
assert np.all(norms > 0)

print("\nBasic embedding checks passed!")

In [ ]:
import json
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer


project_root = Path.cwd()

embedding_path = (
    project_root
    / "data/processed/embeddings/qwen3_embedding_0_6b_records.npz"
)
records_path = (
    project_root
    / "data/processed/embedding_records/sec_10k_embedding_records.jsonl"
)


def as_text(value: object) -> str:
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


records = [
    json.loads(line)
    for line in records_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

with np.load(embedding_path, allow_pickle=False) as data:
    embeddings = data["embeddings"]
    record_ids = [as_text(value) for value in data["record_ids"]]
    model_name = as_text(data["model_name"].item())

assert record_ids == [record["record_id"] for record in records]

model = SentenceTransformer(model_name)


def retrieve(
    query: str,
    ticker: str | None = None,
    k: int = 5,
) -> list[dict]:
    query_embedding = model.encode_query(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0].astype(np.float32)

    scores = embeddings @ query_embedding

    candidate_indices = np.array(
        [
            index
            for index, record in enumerate(records)
            if ticker is None
            or record["metadata"]["ticker"].upper() == ticker.upper()
        ]
    )

    ranked_indices = candidate_indices[
        np.argsort(scores[candidate_indices])[::-1][:k]
    ]

    results = []

    for rank, index in enumerate(ranked_indices, start=1):
        record = records[index]
        metadata = record["metadata"]

        result = {
            "rank": rank,
            "score": float(scores[index]),
            "record_id": record["record_id"],
            "target_chunk_id": record["target_chunk_id"],
            "record_type": record["record_type"],
            "ticker": metadata["ticker"],
            "sec_item": metadata.get("sec_item"),
            "section_title": metadata.get("section_title"),
        }
        results.append(result)

        print(f"\n{'=' * 80}")
        print(result)
        print("\nEmbedding text:")
        print(record["embedding_text"][:600])
        print("\nOriginal document:")
        print(record["document"][:1200])

    return results

In [ ]:
retrieve(
    query=(
        "What operational risks does JPMorgan face from cyberattacks "
        "and failures of technology systems?"
    ),
    ticker="JPM",
)

In [ ]:
retrieve(
    query=(
        "What was JPMorgan Chase's Common Equity Tier 1 capital ratio "
        "at December 31, 2025?"
    ),
    ticker="JPM",
)

In [ ]:
results = retrieve(
    query=(
        "What operational risks does JPMorgan face from cyberattacks "
        "and failures of technology systems?"
    ),
    ticker="JPM",
    k=3,
)

for result in results:
    record_index = record_ids.index(result["record_id"])
    print(f"\nRank {result['rank']} | Score: {result['score']:.4f}")
    print(records[record_index]["document"][:1500])

In [ ]:
from collections import Counter
from importlib.metadata import PackageNotFoundError, version
import json
from pathlib import Path

import numpy as np


root = Path.cwd()

records_path = root / (
    "data/processed/embedding_records/"
    "sec_10k_embedding_records.jsonl"
)
embeddings_path = root / (
    "data/processed/embeddings/"
    "qwen3_embedding_0_6b_records.npz"
)
proxies_path = root / (
    "data/processed/table_proxies/"
    "sec_10k_table_proxies.jsonl"
)

for path in (records_path, embeddings_path, proxies_path):
    if not path.exists():
        raise FileNotFoundError(path)


def load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def get_field(record: dict, field: str):
    if field in record:
        return record[field]

    metadata = record.get("metadata", {})
    if isinstance(metadata, dict):
        return metadata.get(field)

    return None


records = load_jsonl(records_path)
proxies = load_jsonl(proxies_path)

print("===== EMBEDDING RECORDS =====")
print(f"Records: {len(records)}")
print(f"First record fields: {sorted(records[0])}")

record_types = Counter(
    get_field(record, "record_type")
    for record in records
)
tickers = Counter(
    get_field(record, "ticker")
    for record in records
)
counts_by_ticker_and_type = Counter(
    (
        get_field(record, "ticker"),
        get_field(record, "record_type"),
    )
    for record in records
)

print(f"Record types: {dict(sorted(record_types.items()))}")
print(f"Tickers: {dict(sorted(tickers.items()))}")

print("\nCounts by ticker and type:")
for key, count in sorted(counts_by_ticker_and_type.items()):
    print(f"{key}: {count}")

record_ids = [
    str(get_field(record, "record_id"))
    for record in records
]
target_chunk_ids = [
    str(get_field(record, "target_chunk_id"))
    for record in records
]

print(f"\nUnique record IDs: {len(set(record_ids))}")
print(
    "Unique target chunk IDs: "
    f"{len(set(target_chunk_ids))}"
)

print("\n===== TABLE PROXIES =====")
proxy_versions = Counter(
    proxy.get("proxy_version", "<missing>")
    for proxy in proxies
)

print(f"Proxies: {len(proxies)}")
print(f"Proxy versions: {dict(proxy_versions)}")
print(f"First proxy fields: {sorted(proxies[0])}")

print("\n===== NPZ =====")
with np.load(embeddings_path, allow_pickle=False) as archive:
    print(f"NPZ keys: {archive.files}")

    for key in archive.files:
        array = archive[key]
        print(
            f"{key}: shape={array.shape}, "
            f"dtype={array.dtype}"
        )

    vector_keys = [
        key
        for key in archive.files
        if archive[key].ndim == 2
        and np.issubdtype(archive[key].dtype, np.floating)
    ]

    if len(vector_keys) != 1:
        raise ValueError(
            f"Expected one embedding matrix, found: {vector_keys}"
        )

    vector_key = vector_keys[0]
    embeddings = archive[vector_key]

    id_keys = [
        key
        for key in archive.files
        if "id" in key.lower()
        and archive[key].ndim == 1
        and len(archive[key]) == len(records)
    ]

    print(f"Embedding array key: {vector_key}")
    print(f"Possible ID keys: {id_keys}")

    norms = np.linalg.norm(embeddings, axis=1)

    print(f"Embedding count: {embeddings.shape[0]}")
    print(f"Embedding dimension: {embeddings.shape[1]}")
    print(f"Minimum norm: {norms.min():.8f}")
    print(f"Maximum norm: {norms.max():.8f}")
    print(f"NaN values: {int(np.isnan(embeddings).sum())}")
    print(f"Inf values: {int(np.isinf(embeddings).sum())}")

    if id_keys:
        npz_ids = archive[id_keys[0]].astype(str).tolist()
        print(
            "NPZ and JSONL ID order match: "
            f"{npz_ids == record_ids}"
        )

print("\n===== VERIFICATION ENVIRONMENT =====")
packages = [
    "numpy",
    "sentence-transformers",
    "transformers",
    "torch",
    "huggingface-hub",
]

for package in packages:
    try:
        package_version = version(package)
    except PackageNotFoundError:
        package_version = "<not installed>"

    print(f"{package}: {package_version}")

In [ ]:
import json
import re
from pathlib import Path
from typing import Any


records_path = Path(
    "data/processed/embedding_records/"
    "sec_10k_embedding_records.jsonl"
)

records = [
    json.loads(line)
    for line in records_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]


def get_field(
    record: dict[str, Any],
    field: str,
) -> Any:
    if field in record:
        return record[field]

    metadata = record.get("metadata", {})

    if isinstance(metadata, dict):
        return metadata.get(field)

    return None


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


answer_pattern = re.compile(
    r"(?<![\d.])14\.6\s*%?(?!\d)",
    flags=re.IGNORECASE,
)

matches: list[dict[str, str]] = []

for record in records:
    ticker = str(get_field(record, "ticker") or "").upper()
    document = normalize_text(str(record["document"]))

    if ticker != "JPM":
        continue

    if "2025" not in document:
        continue

    if "cet1" not in document.lower():
        continue

    if not answer_pattern.search(document):
        continue

    matches.append(
        {
            "record_id": str(record["record_id"]),
            "target_chunk_id": str(
                record["target_chunk_id"]
            ),
            "record_type": str(record["record_type"]),
            "document": document,
        }
    )


print(f"Direct 14.6% candidates: {len(matches)}")

for index, match in enumerate(matches, start=1):
    answer_match = answer_pattern.search(match["document"])
    assert answer_match is not None

    start = max(0, answer_match.start() - 700)
    end = min(
        len(match["document"]),
        answer_match.end() + 700,
    )

    print(f"\n===== CANDIDATE {index} =====")
    print(f"Record ID: {match['record_id']}")
    print(
        "Target chunk ID: "
        f"{match['target_chunk_id']}"
    )
    print(f"Type: {match['record_type']}")
    print(
        "Evidence:\n"
        f"{match['document'][start:end]}"
    )

In [ ]:
import json
import re
from pathlib import Path
from typing import Any


records_path = Path(
    "data/processed/embedding_records/"
    "sec_10k_embedding_records.jsonl"
)

records = [
    json.loads(line)
    for line in records_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]


def get_field(
    record: dict[str, Any],
    field: str,
) -> Any:
    if field in record:
        return record[field]

    metadata = record.get("metadata", {})

    if isinstance(metadata, dict):
        return metadata.get(field)

    return None


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


definition_patterns = (
    "operational risk is",
    "operational risk refers",
    "operational risk includes",
)

matches: dict[str, dict[str, str]] = {}

for record in records:
    ticker = str(get_field(record, "ticker") or "").upper()
    record_type = str(record.get("record_type", ""))
    document = normalize_text(str(record["document"]))
    document_lower = document.lower()

    if ticker != "JPM" or record_type != "text":
        continue

    matched_pattern = next(
        (
            pattern
            for pattern in definition_patterns
            if pattern in document_lower
        ),
        None,
    )

    if matched_pattern is None:
        continue

    target_chunk_id = str(record["target_chunk_id"])

    matches.setdefault(
        target_chunk_id,
        {
            "record_id": str(record["record_id"]),
            "target_chunk_id": target_chunk_id,
            "matched_pattern": matched_pattern,
            "document": document,
        },
    )


print(f"Operational-risk candidates: {len(matches)}")

for index, match in enumerate(matches.values(), start=1):
    document = match["document"]
    pattern_position = document.lower().find(
        match["matched_pattern"]
    )

    start = max(0, pattern_position - 500)
    end = min(len(document), pattern_position + 2000)

    print(f"\n===== CANDIDATE {index} =====")
    print(f"Record ID: {match['record_id']}")
    print(f"Target chunk ID: {match['target_chunk_id']}")
    print(f"Matched pattern: {match['matched_pattern']}")
    print(f"Evidence:\n{document[start:end]}")

In [ ]:
import json
from pathlib import Path

path = Path(
    "data/experiments/jpm_sec2md/"
    "structure_aware/table_parents.jsonl"
)

parents = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

with_grids = [
    parent
    for parent in parents
    if parent.get("cell_matrices")
]

print("Table parents:", len(parents))
print("Parents with grids:", len(with_grids))
print("Parents without grids:", len(parents) - len(with_grids))

for parent in with_grids[:3]:
    matrix = parent["cell_matrices"][0]

    print("\nLogical table:", parent["logical_table_id"])
    print("Element:", parent["metadata"]["table_element_id"])
    print("Shape:", len(matrix), "x", max(map(len, matrix), default=0))

    for row in matrix[:8]:
        print(row)

In [ ]:
import json
from collections import Counter
from pathlib import Path

path = Path(
    "data/experiments/jpm_sec2md/"
    "structure_aware/table_parents.jsonl"
)

parents = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

counts = Counter(parent["table_type"] for parent in parents)

print("Table types:")
for table_type, count in counts.most_common():
    print(f"{table_type}: {count}")

In [ ]:
import json
from collections import Counter
from pathlib import Path

path = Path(
    "data/experiments/jpm_sec2md/"
    "structure_aware/embedding_records.jsonl"
)

records = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

locators = [
    record
    for record in records
    if record["record_type"] == "table_locator"
]

print("Eligible records:", len(records))
print("Table locators:", len(locators))
print(
    "Locator scopes:",
    Counter(
        locator["metadata"]["locator_scope"]
        for locator in locators
    ),
)

for locator in locators[:5]:
    print("\n", locator["document"])
    print("Evidence:", locator["metadata"]["evidence_ref"])

In [ ]:
import json
from collections import Counter
from pathlib import Path

path = Path(
    "data/experiments/jpm_sec2md/"
    "structure_aware/embedding_records.jsonl"
)

records = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

locators = [
    record
    for record in records
    if record["record_type"] == "table_locator"
]

row_locators = [
    locator
    for locator in locators
    if locator["metadata"]["locator_scope"] == "row"
]

invalid_bindings = [
    locator
    for locator in row_locators
    if len(locator["metadata"]["column_paths"])
    != len(locator["metadata"]["cell_coordinates"])
]

group_counts = Counter(
    locator["metadata"]["locator_group_id"]
    for locator in row_locators
)

fallback_parts = [
    locator
    for locator in locators
    if locator["metadata"]["locator_part_count"] > 1
]

print("Eligible records:", len(records))
print("Table locators:", len(locators))
print("Row locators:", len(row_locators))
print("Invalid column-coordinate bindings:", len(invalid_bindings))
print(
    "Rows split into column groups:",
    sum(count > 1 for count in group_counts.values()),
)
print(
    "Maximum column groups for one row:",
    max(group_counts.values(), default=0),
)
print("Text fallback parts:", len(fallback_parts))

for locator in row_locators:
    metadata = locator["metadata"]

    if metadata["column_group_count"] > 1:
        print("\n", locator["document"])
        print("Group:", metadata["column_group_index"] + 1)
        print("of:", metadata["column_group_count"])
        print("Coordinates:", metadata["cell_coordinates"])
        break